## ESPnet TTS Demonstracija

Šie skriptai demonstruoja kaip galima sintezuoti lietuvišką garsą, kai lokaliai turime:
1. Akustinio modelio archyvą
2. Vokoderio archyvą
   
### 0. Inicializuokite pagalbines funkcijas

In [ ]:
# Inicializuokite pagalbines funkcijas

from espnet2.bin.tts_inference import Text2Speech
import soundfile as sf
from IPython.display import Audio, HTML
from espnet2.main_funcs.pack_funcs import unpack, find_path_and_change_it_recursive
import tarfile
from pathlib import Path
from typing import Dict
import yaml

cache_dir = "./.cache/"
device = "cpu"
#device = "cuda" # uncomment if you have a GPU and want to use it

def _safe_extract_tar(tar: tarfile.TarFile, path: Path):
    base = path.resolve()

    for member in tar.getmembers():
        member_path = (path / member.name).resolve()
        if not str(member_path).startswith(str(base)):
            raise Exception("Unsafe tar archive")

    tar.extractall(path)

def find_vocoder(search_dir: Path) -> str:
    for file in search_dir.rglob("*.pkl"):
        # print(f"found {str(file)}")
        return str(file)

    raise FileNotFoundError(f"No *.pkl file found in the {search_dir}")    


def get_dict_from_cache(meta: Path) -> Dict[str, str]:
    if not meta.exists():
        raise FileNotFoundError(f"No meta.yaml found in the {meta}")
        
    outpath = meta.parent
    with meta.open("r", encoding="utf-8") as f:
        d = yaml.safe_load(f)
        yaml_files = d["yaml_files"]
        files = d["files"]

        for _, value in yaml_files.items():
            if not (outpath / value).exists():
                raise FileNotFoundError(f"No {outpath / value} found")
            with (outpath / value).open("r", encoding="utf-8") as f:
                d_int = yaml.safe_load(f)
            for path in outpath.rglob("*"):
                if path.is_file():
                    relative_path = path.relative_to(outpath)
                    d_int = find_path_and_change_it_recursive(d_int, str(relative_path), str(path))    
            with (outpath / value).open("w", encoding="utf-8") as f:
                yaml.safe_dump(d_int, f)
        retval = {}
        for key, value in list(yaml_files.items()) + list(files.items()):
            if not (outpath / value).exists():
                raise FileNotFoundError(f"No {outpath / value} found")
            retval[key] = str(outpath / value)
        return retval
        

def extract_archive_cached(archive_path: str, cache_dir: str) -> Path:
    archive_path = Path(archive_path)
    if archive_path.is_dir():
        return archive_path  # already extracted
    
    cache_dir = Path(cache_dir)

    stat = archive_path.stat()

    # fast cache key (no hashing)
    key = f"{archive_path.name}_{stat.st_size}_{int(stat.st_mtime)}"

    extract_path = cache_dir / key
    marker = extract_path / ".done"

    if marker.exists():
        return extract_path

    extract_path.mkdir(parents=True, exist_ok=True)

    if archive_path.name.endswith(".tar.gz"):
        with tarfile.open(archive_path) as tar:
            _safe_extract_tar(tar, extract_path)
    else:
        raise ValueError(f"Unsupported archive format: {archive_path}")

    marker.touch()
    return extract_path
    

def init_tts(am_file=None, vocoder_file=None):
    if not vocoder_file:
        vocoder_file = None # make sure it's None, not empty string
    else:
        vocoder_dir = extract_archive_cached(archive_path=vocoder_file, cache_dir=cache_dir)
        vocoder_file = find_vocoder(vocoder_dir)
        print(f"vocoder_file: {vocoder_file}")

    print(f"\n===================================\nAM      = {am_file}")
    print(f"Vocoder = {vocoder_file if vocoder_file else 'Griffin-Lim'}")
    print(f"===================================")

    am_path = Path(am_file)
    if am_path.is_dir():
        meta = am_path / "meta.yaml"
        kwargs = get_dict_from_cache(meta=meta)
    else:    
        kwargs = unpack(input_archive = am_file, outpath = cache_dir)
    print (kwargs)
    
    tts = Text2Speech.from_pretrained(
        vocoder_file=vocoder_file, 
        device=device,
        **kwargs
    )
    return tts

## workaround: fix parallel_wavegan for python3.12
import scipy.signal.windows
import sys
sys.modules['scipy.signal'].kaiser = scipy.signal.windows.kaiser     

print ("ESPnet inicializuotas")    

### 1. Modelis

Nurodykite kelią iki Akustinio modelio ir Vokoderio

In [ ]:
# Nurodykite kelią iki akustinio modelio zip failo
# arba iki direktorijos, jei parsisiuntėte iš HuggingFace
# pvz.:  git clone https://huggingface.co/VSSA-SDSA/sing-agn.fastspeech2.v01
am_zip_file_or_dir="agn.tts_train_fastspeech2_raw_phn_espeak_ng_lt_train.loss.ave.zip"
am_zip_file_or_dir="./sing-agn.fastspeech2.v01"

# Nurodykite kelią iki vokoderio tar gz failo
# arba iki direktorijos, jei parsisiuntėte iš HuggingFace
# pvz.: git clone https://huggingface.co/VSSA-SDSA/sing-agn.vocoder.style_melgan.v01

# arba palikite tuščią, jei norite naudoti Griffin-Lim algoritmą
vocoder_tar_gz_file_or_dir="agn.style.v01-600000.tar.gz"
vocoder_tar_gz_file_or_dir="sing-agn.vocoder.style_melgan.v01"
# vocoder_tar_gz_file_or_dir=None

# bus išskleisti į './.cache/' katalogą"
print (f"AM zip failas  : {am_zip_file_or_dir}")
print (f"Vocoder failas : {vocoder_tar_gz_file_or_dir}")

# Inicializuokite TTS modelį
tts = init_tts(am_file=am_zip_file_or_dir, vocoder_file=vocoder_tar_gz_file_or_dir)
print (f"\n\nREADY: modelis įkeltas\n") 

### 2. Sintezavimas

Pakoreguokite/įveskite sakinius, kuriuos norite perskaityti

In [ ]:
### Kai kurie tekstai paimti iš lrt.lt
texts = ["Sveiki, aš naujas lietuviškas balsas.", 
         "Aš esu labai gerai įrašytas.", 
         "Kaip Jums patinku?", 
         "Pernai maitinimo sektorių sukrėtė dešimtmečius veikusių restoranų ir kavinių bankrotai.", 
         "Šiomis dienomis orai Lietuvoje ims šilti, kris šlapdriba, sniegas, reikės pasisaugoti lijundros.", 
         "Tai savo „Facebook“ paskyroje sekmadienį vakare pranešė Ukrainos pirmasis vicepremjeras ir energetikos ministras Denysas Šmyhalis, skelbia „Ukrinform“."
        ]
for i, text in enumerate(texts):
    print(f"\n==========================================================================\nText = {text}")
    wav = tts(text)["wav"]
    filename = f"output_{i}.wav"
    sf.write(filename, wav.cpu().numpy(), tts.fs)
    display(Audio(filename))
